In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
import exciting_environments as excenvs

import dmpe
from dmpe.models.models import NeuralEulerODEPendulum, NeuralODEPendulum, NeuralEulerODE, NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.models.model_training import ModelTrainer
from dmpe.excitation.excitation_utils import loss_function, Exciter

from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation, update_density_estimate_multiple_observations, DensityEstimate
)
from dmpe.utils.signals import aprbs
from dmpe.evaluation.plotting_utils import (
    plot_sequence, append_predictions_to_sequence_plot, plot_sequence_and_prediction, plot_model_performance
)
from dmpe.evaluation.experiment_utils import (
    get_experiment_ids, load_experiment_results, quick_eval, evaluate_experiment_metrics, evaluate_algorithm_metrics, load_all_experiment_results
)
from dmpe.utils.density_estimation import select_bandwidth
from dmpe.evaluation.experiment_utils import default_jsd, default_ae, default_mcudsa, default_ksfc

In [ ]:
from dmpe.evaluation.model_evaluation import ModelEvaluator, EnvWrapper
from dmpe.evaluation.data_evaluation import DataEvaluator
from dmpe.evaluation.utils import default_constraint_function

In [ ]:
from dmpe.data_management import DataPaths
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.evaluation.imperfect_pm_dmpe import adapt_static_params

- iterate over the results
- evaluate data quality
- reconstruct env from given parameters
- evaluate model quality

In [ ]:
sys_name = "cart_pole"

In [ ]:
if sys_name == "fluid_tank":
    env, penalty_function, featurize, env_params = setup_fluid_tank_env()
elif sys_name == "pendulum":
    env, penalty_function, featurize, env_params = setup_pendulum_env()
elif sys_name == "cart_pole":
    env, penalty_function, featurize, env_params = setup_cart_pole_env()

In [ ]:
results = load_all_experiment_results(DataPaths().imperfect_pm_dmpe_experiments / sys_name, None)

In [ ]:
obs_dim = env.reset(env.env_properties)[0].shape[-1]
act_dim = env.action_dim

wrapped_env = EnvWrapper(env, featurize)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=obs_dim,
    act_dim=act_dim,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

data_evaluator = DataEvaluator(
    constraint_function=default_constraint_function,
    data_dim=obs_dim,# + act_dim,
    points_per_dim=points_per_dim,
)

In [ ]:
jsds = []
model_errors = []

for result in tqdm(results):

    observations = result["observations"]
    actions = result["actions"]
    
    new_static_params = env.StaticParams(**result["params"]["model_params"]["new_static_params"])
    model = adapt_static_params(env, new_static_params)
    jsd_value = data_evaluator.default_metrics["jsd"](observations)
    # jsd_value = data_evaluator.default_metrics["jsd"](jnp.concatenate([observations[:-1], actions], axis=-1))    
    _, model_error = model_evaluator.default_metrics["pred_comp"](wrapped_env, EnvWrapper(model, featurize))

    jsds.append(jsd_value)
    model_errors.append(model_error)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

ax.scatter(
    jsds, model_errors, s=25, marker="x")

ax.set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
ax.grid(True)
ax.set_yscale("log")
ax.set_xscale("log")

ax.set_ylim(10**-6, 10**-2)

plt.show()